In [1]:
# Standard library imports
from argparse import ArgumentParser
import os, sys
THIS_DIR = os.path.abspath('')
PARENT_DIR = os.path.dirname(os.path.abspath(''))
sys.path.append(PARENT_DIR)

# Third party imports
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint
from torchdiffeq import odeint
from torchvision import utils
import matplotlib.pyplot as plt
import numpy as np

# local application imports
from lag_caVAE.lag import Lag_Net
from lag_caVAE.nn_models import MLP_Encoder, MLP, MLP_Decoder, PSD
from hyperspherical_vae.distributions import VonMisesFisher
from hyperspherical_vae.distributions import HypersphericalUniform
from utils import arrange_data, from_pickle, my_collate, ImageDataset, HomoImageDataset
from examples.pend_lag_cavae_trainer import Model as Model_lag_cavae
from ablations.ablation_pend_MLPdyna_cavae_trainer import Model as Model_MLPdyna_cavae
from ablations.ablation_pend_lag_vae_trainer import Model as Model_lag_vae
from ablations.ablation_pend_lag_caAE_trainer import Model as Model_lag_caAE
from ablations.HGN import Model as Model_HGN

seed_everything(0)
%matplotlib inline
DPI = 600

Global seed set to 0
Global seed set to 0
Global seed set to 0
Global seed set to 0
Global seed set to 0


In [10]:
checkpoint_path = os.path.join(PARENT_DIR, 
                               'results', 
                               'pend', 
                               'pend-lag-cavae-T_p=4-epoch=653-step=26159.ckpt')
model_lag_cavae = Model_lag_cavae.load_from_checkpoint(checkpoint_path)

checkpoint_path = os.path.join(PARENT_DIR, 
                               'checkpoints', 
                               'ablation-pend-MLPdyna-cavae-T_p=4-epoch=919.ckpt')
model_MLPdyna_cavae = Model_MLPdyna_cavae.load_from_checkpoint(checkpoint_path)

checkpoint_path = os.path.join(PARENT_DIR, 
                               'checkpoints', 
                               'ablation-pend-lag-vae-T_p=4-epoch=916.ckpt')
model_lag_vae = Model_lag_vae.load_from_checkpoint(checkpoint_path)

checkpoint_path = os.path.join(PARENT_DIR, 
                               'checkpoints', 
                               'ablation-pend-lag-caAE-T_p=4-epoch=778.ckpt')
model_lag_caAE = Model_lag_caAE.load_from_checkpoint(checkpoint_path)

checkpoint_path = os.path.join(PARENT_DIR, 
                               'checkpoints', 
                               'baseline-pend-HGN-T_p=4-epoch=1543.ckpt')
model_HGN = Model_HGN.load_from_checkpoint(checkpoint_path)

In [11]:
# Load train data, prepare for plotting prediction
data_path=os.path.join(PARENT_DIR, 'datasets', 'pendulum-gym-image-dataset-train.pkl')
train_dataset = HomoImageDataset(data_path, T_pred=4)
# prepare model
model_lag_cavae.t_eval = torch.from_numpy(train_dataset.t_eval)
model_lag_cavae.hparams.annealing = False
model_MLPdyna_cavae.t_eval = torch.from_numpy(train_dataset.t_eval)
model_lag_vae.t_eval = torch.from_numpy(train_dataset.t_eval)
model_lag_caAE.t_eval = torch.from_numpy(train_dataset.t_eval)
model_HGN.t_eval = torch.from_numpy(train_dataset.t_eval)
model_HGN.step = 3 ; model_HGN.alpha = 1

In [12]:
lag_cavae_train_loss = []
MLPdyna_cavae_train_loss = []
lag_vae_train_loss = []
lag_caAE_train_loss = []

for i in range(len(train_dataset.x)):
    batch = (torch.from_numpy(train_dataset.x[i]), torch.from_numpy(train_dataset.u[i]))
    lag_cavae_train_loss.append(model_lag_cavae.training_step(batch, 0)['log']['recon_loss'].item())
    MLPdyna_cavae_train_loss.append(model_MLPdyna_cavae.training_step(batch, 0)['log']['recon_loss'].item())
    lag_vae_train_loss.append(model_lag_vae.training_step(batch, 0)['log']['recon_loss'].item())
    lag_caAE_train_loss.append(model_lag_caAE.training_step(batch, 0)['log']['recon_loss'].item())


In [5]:
HGN_train_loss = []
train_dataset.u_idx = 0
dataLoader = DataLoader(train_dataset, batch_size=256, shuffle=False, collate_fn=my_collate)
for batch in dataLoader:
    HGN_train_loss.append(model_HGN.training_step(batch, 0)['log']['recon_loss'].item())

ValueError: Expected parameter scale (Tensor of shape (256, 24, 16, 16)) of distribution Normal(loc: torch.Size([256, 24, 16, 16]), scale: torch.Size([256, 24, 16, 16])) to satisfy the constraint GreaterThan(lower_bound=0.0), but found invalid values:
tensor([[[[-8.6320e-01, -8.6317e-01, -8.6301e-01,  ..., -8.6313e-01,
           -8.6324e-01, -8.6320e-01],
          [-8.6314e-01, -8.6291e-01, -8.6199e-01,  ..., -8.6155e-01,
           -8.6322e-01, -8.6321e-01],
          [-8.6287e-01, -8.6192e-01, -8.5851e-01,  ..., -8.5346e-01,
           -8.6253e-01, -8.6322e-01],
          ...,
          [-8.6309e-01, -8.6071e-01, -8.5126e-01,  ..., -7.0495e-01,
           -6.4029e-01, -7.2393e-01],
          [-8.6320e-01, -8.6198e-01, -8.5562e-01,  ..., -7.1453e-01,
           -6.5371e-01, -7.3271e-01],
          [-8.6323e-01, -8.6267e-01, -8.5897e-01,  ..., -6.8128e-01,
           -6.5645e-01, -7.5397e-01]],

         [[-1.0344e+00, -1.0342e+00, -1.0338e+00,  ..., -1.0334e+00,
           -1.0341e+

In [13]:
# Load data, prepare for plotting prediction
data_path=os.path.join(PARENT_DIR, 'datasets', 'pendulum-gym-image-dataset-test.pkl')
test_dataset = HomoImageDataset(data_path, 4)

In [14]:
lag_cavae_test_loss = []
MLPdyna_cavae_test_loss = []
lag_vae_test_loss = []
lag_caAE_test_loss = []

for i in range(len(train_dataset.x)):
    batch = (torch.from_numpy(test_dataset.x[i]), torch.from_numpy(test_dataset.u[i]))
    lag_cavae_test_loss.append(model_lag_cavae.training_step(batch, 0)['log']['recon_loss'].item())
    MLPdyna_cavae_test_loss.append(model_MLPdyna_cavae.training_step(batch, 0)['log']['recon_loss'].item())
    lag_vae_test_loss.append(model_lag_vae.training_step(batch, 0)['log']['recon_loss'].item())
    lag_caAE_test_loss.append(model_lag_caAE.training_step(batch, 0)['log']['recon_loss'].item())

In [15]:
HGN_test_loss = []
train_dataset.u_idx = 0
dataLoader = DataLoader(test_dataset, batch_size=256, shuffle=False, collate_fn=my_collate)
for batch in dataLoader:
    HGN_test_loss.append(model_HGN.training_step(batch, 0)['log']['recon_loss'].item())

ValueError: Expected parameter scale (Tensor of shape (256, 24, 16, 16)) of distribution Normal(loc: torch.Size([256, 24, 16, 16]), scale: torch.Size([256, 24, 16, 16])) to satisfy the constraint GreaterThan(lower_bound=0.0), but found invalid values:
tensor([[[[-8.6320e-01, -8.6317e-01, -8.6302e-01,  ..., -8.6305e-01,
           -8.6318e-01, -8.6317e-01],
          [-8.6314e-01, -8.6291e-01, -8.6203e-01,  ..., -8.5999e-01,
           -8.6203e-01, -8.6260e-01],
          [-8.6288e-01, -8.6188e-01, -8.5852e-01,  ..., -8.4167e-01,
           -8.5326e-01, -8.5774e-01],
          ...,
          [-8.6314e-01, -8.6194e-01, -8.5784e-01,  ..., -6.7161e-01,
           -6.1031e-01, -5.9712e-01],
          [-8.6321e-01, -8.6284e-01, -8.6107e-01,  ..., -7.4588e-01,
           -7.1647e-01, -7.1182e-01],
          [-8.6322e-01, -8.6316e-01, -8.6274e-01,  ..., -8.1293e-01,
           -8.0895e-01, -8.1112e-01]],

         [[-1.0344e+00, -1.0342e+00, -1.0338e+00,  ..., -1.0329e+00,
           -1.0338e+

In [16]:
scale = 32*32*5
print(f'lag_cavae: train: {np.mean(lag_cavae_train_loss)/scale}, test: {np.mean(lag_cavae_test_loss)/scale}')
print(f'MLPdyna_cavae: train: {np.mean(MLPdyna_cavae_train_loss)/scale}, test: {np.mean(MLPdyna_cavae_test_loss)/scale}')
print(f'lag_vae: train: {np.mean(lag_vae_train_loss)/scale}, test: {np.mean(lag_vae_test_loss)/scale}')
print(f'lag_caAE: train: {np.mean(lag_caAE_train_loss)/scale}, test: {np.mean(lag_caAE_test_loss)/scale}')
print(f'HGN: train: {np.mean(HGN_train_loss)/scale}, test: {np.mean(HGN_test_loss)/scale}')

lag_cavae: train: 0.00195624265819788, test: 0.001991308219730854
MLPdyna_cavae: train: 0.0018252068385481834, test: 0.0018656634911894798
lag_vae: train: 0.002406070977449417, test: 0.0025191659852862357
lag_caAE: train: 0.001860453225672245, test: 0.00189580075442791
HGN: train: nan, test: nan
